# Autonomous Movie Studio — Real Qwen GPU Validation (Colab)

> **Repo-local validation notebook for the real-LLM pipeline.**
> Runs the actual Qwen director + script writer (Transformers / CUDA) that the
> orchestrator uses under **strict GPU mode**, and records what actually executed
> in `provider_manifest.json`.

📅 Status marker: this notebook is auto-regenerated by the repo tooling.


## What "validated by a REAL LLM" means

The orchestrator is normally driven by stubs/mocks so CI runs anywhere.
`REQUIRE_REAL_LLM=true` flips it into **strict GPU mode**:

- `require_cuda()` fails hard when CUDA is unavailable (no CPU fallback).
- The director + script stages refuse `mock`/deterministic providers.
- A `provider_manifest.json` records exactly which provider/model executed and how
  long model load + generation took.

This notebook **reproduces the orchestrator's strict-mode code paths** against a real
video, so we can prove on a T4 GPU that:
1. CUDA torch is present (nvidia-smi / `torch.cuda`).
2. The Qwen producer actually calls Transformers (its `generate_text` path, not a stub).
3. `provider_manifest.json` reports `director_real_generation: true`,
   `script_real_generation: true`, with model `Qwen/Qwen3-4B-Instruct-2507` and CUDA device.


## How to run

1. Open this notebook in Colab with a **GPU** runtime:
   `Runtime → Change runtime type → T4 GPU`.
2. **Edit cell 4** to point at your repo (private repos: use a token URL).
3. Run all cells top-to-bottom (`Runtime → Run all`).
4. Open `data/ColabValidation/provider_manifest.json` and verify the strict flags.

> It is installed/run in a fresh Colab VM (`/content`), so it git-clones the repo;
> changes you make while testing won't affect your working repo.

### If you hit "out of memory"
- The director and script stages **share one loaded model** (a class-level cache), so
  the model is only loaded **once** — not twice.
- Loading leaves VRAM headroom (`QWEN_VRAM_RESERVE_GB`) and streams from disk
  (`low_cpu_mem_usage`, `device_map=auto`) so it won't crash on Colab's RAM.
- Still tight? Switch to **4-bit** in cell 6: uncomment
  `DIRECTOR_DTYPE=4bit` and `SCRIPT_DTYPE=4bit` (~4GB VRAM total).
- Generations are released (`torch.cuda.empty_cache()`) between stages.


In [ ]:
# ============================================================
# [EDITME] Your repository
REPO_URL = "https://github.com/<OWNER>/<REPO_NAME>.git"
REPO_DIR = "automovie-studio"

# --- T4 should be fully available; 4B fits in fp16 easily ---
!nvidia-smi

import os, glob, sys, json, shutil, subprocess

# Operate from a stable base dir so re-running this cell NEVER clones the repo
# inside itself (no automovie-studio/automovie-studio nesting).
BASE = "/content"
if os.path.isdir(BASE) and os.getcwd() != BASE:
    os.chdir(BASE)
os.chdir(BASE)

if not os.path.isdir(REPO_DIR):
    print("Cloning", REPO_URL)
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    # Hard reset guarantees the newest code even if a previous pull/merge left
    # the working tree stale (safe: this is a throwaway validation VM).
    print("Repo dir present; hard-resetting to origin/main")
    !git -C {os.path.join(BASE, REPO_DIR)} fetch origin --depth 1
    !git -C {os.path.join(BASE, REPO_DIR)} reset --hard origin/main

os.chdir(os.path.join(BASE, REPO_DIR))
head = subprocess.run(
    ["git", "log", "--oneline", "-1"], capture_output=True, text=True
).stdout.strip()
print("Repo:", os.getcwd())
print("HEAD:", head)
assert "fdaa732" in head or "477ea8f" in head or "fb2f260" in head, (
    "Repo HEAD is unexpectedly old: " + head
)
print("[PASS] Repo is at the expected commit.")


In [ ]:
# ============================================================
# Idempotent dependency setup (ffmpeg, CUDA torch, transformers,
# accelerate, whisper, pyscenedetect)
!bash scripts/colab_setup.sh


In [ ]:
# ============================================================
# Env: strict GPU mode + colab-gpu profile (exactly what the
# orchestrator reads at runtime)
import os
os.environ["REQUIRE_REAL_LLM"] = "true"
os.environ["STUDIO_PROFILE"]   = "colab-gpu"
os.environ["DIRECTOR_PROVIDER"] = "qwen"
os.environ["DIRECTOR_MODEL"]   = "Qwen/Qwen3-4B-Instruct-2507"
os.environ["SCRIPT_PROVIDER"]  = "qwen"
os.environ["SCRIPT_MODEL"]     = "Qwen/Qwen3-4B-Instruct-2507"

# VRAM headroom left free on the GPU during model load (prevents CUDA OOM).
os.environ["QWEN_VRAM_RESERVE_GB"] = "2.5"
# If you still hit out-of-memory, load the model in 4-bit instead of fp16:
# os.environ["DIRECTOR_DTYPE"] = "4bit"
# os.environ["SCRIPT_DTYPE"]   = "4bit"

import sys
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
sys.path.insert(0, os.getcwd())

from utils import strict
print("strict_mode_enabled():", strict.strict_mode_enabled())

from utils.doctor import run_checks
report = run_checks()
for k in ("torch", "transformers", "accelerate", "require_real_llm",
          "strict_gpu_mode", "active_profile",
          "director_provider", "director_model",
          "script_provider", "script_model"):
    v = report.get(k)
    if k == "torch" and isinstance(v, dict):
        v = {kk: vv for kk, vv in v.items() if kk != "vision" }
    print(f"{k:.<20} {v}")


In [ ]:
# ============================================================
# Pretty doctor output (same as `python main.py doctor`)
try:
    from utils.doctor import print_report
    print_report(report)
except Exception as e:
    print("print_report not available:", e)
    print(json.dumps(report, indent=2, default=str)[:2000])


In [ ]:
# ============================================================
# Strict-mode validation gate (parity with doctor's STRICT section)
print("REQUIRE_REAL_LLM   :", report.get("require_real_llm"))
print("Profile            :", report.get("active_profile"))
print("Strict mode        :", report.get("strict_gpu_mode"))
print("Strict prerequisites OK:", report.get("strict_gpu_ok"))
print("Director           :", report.get("director_provider"), "/", report.get("director_model"))
print("Script             :", report.get("script_provider"), "/", report.get("script_model"))

assert report.get("require_real_llm") is True
assert report.get("strict_gpu_mode") is True
assert report.get("strict_gpu_ok") is True, "CUDA + transformers + accelerate must be present"
assert report.get("director_provider") == "qwen"
assert report.get("script_provider") == "qwen"
print("[PASS] Strict preconditions satisfied on this runtime.")


In [ ]:
# ============================================================
# CUDA details (the device Qwen will run on)
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("compute cap:", f"{props.major}.{props.minor}", "| vram GB:", round(props.total_memory / 1e9, 1))


In [ ]:
# ============================================================
# Build a REAL project: register a video, then run the real
# transcription + scene-indexing adapters used by the orchestrator.
PROJECT_ID = "ColabValidation"
project_dir = os.path.join("data", PROJECT_ID)
os.makedirs(project_dir, exist_ok=True)

# Find an existing short video, else synthesize a tiny clip.
# Skip our own previous validation clip so a stale silent sample isn't reused.
candidates = sorted(glob.glob("data/**/*.mp4", recursive=True)) + \
             sorted(glob.glob("tests/**/*.mp4", recursive=True))
candidates = [c for c in candidates if "ColabValidation" not in c]
video = None
for c in candidates:
    try:
        dur = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration",
                              "-of","csv=p=0", c], capture_output=True, text=True).stdout.strip()
        if float(dur or 0) >= 8:
            video = c; break
    except Exception:
        continue
if not video:
    print("No repo video found — synthesizing a spoken test clip")
    clip = "data/ColabValidation/sample.mp4"
    # Drop any stale clip from a previous run so we always get fresh content.
    for stale in ("data/ColabValidation/sample.mp4", "data/ColabValidation/voiceover.mp3"):
        if os.path.exists(stale):
            os.remove(stale)
    voiceover = (
        "Why do we keep rewatching the same films? Every repeat viewing "
        "exposes a detail the first watch hides, a gesture, a cut, a line "
        "that reframes the whole story. This film builds its meaning out of "
        "hidden repetitions, and a close reading shows the structure itself "
        "is the argument."
    )
    ok = False
    try:
        import asyncio
        import edge_tts

        async def _speak(text, out):
            await edge_tts.Communicate(text, "en-US-JennyNeural").save(out)

        asyncio.run(_speak(voiceover, "data/ColabValidation/voiceover.mp3"))
        ok = True
    except Exception as e:
        print("edge-tts unavailable:", e)
    if ok:
        !ffmpeg -y -f lavfi -i testsrc=duration=15:size=640x360:rate=24 \
                -i "data/ColabValidation/voiceover.mp3" -c:v libx264 \
                -preset veryfast -c:a aac -shortest "data/ColabValidation/sample.mp4"
    else:
        print("Falling back to silent testsrc clip (concepts will be thin)")
        !ffmpeg -y -f lavfi -i testsrc=duration=12:size=640x360:rate=24 \
                -f lavfi -i sine=frequency=440:duration=12 \
                -c:v libx264 -preset veryfast -c:a aac "data/ColabValidation/sample.mp4"
    video = clip
print("Source:", video)

# project_meta.json mirrors what start_pipeline reads
meta = {"title": "Colab Validation Cut", "source_path": video, "duration_sec": None}
with open(os.path.join(project_dir, "project_meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

from transcription.adapter import transcribe
transcribe(project_dir, video)

from scene_indexing.adapter import build_scene_cards
build_scene_cards(project_dir, video)

tdf = os.path.join(project_dir, "transcripts", "transcript.json")
sdf = os.path.join(project_dir, "scenes", "scene_index.json")
print("transcript:", tdf, os.path.exists(tdf))
print("scene index:", sdf, os.path.exists(sdf))


In [ ]:
# ============================================================
# CREATIVE DIRECTOR — REAL Qwen on CUDA
# Mirrors app/orchestrator.start_pipeline() strict path exactly.
import json, time
from pathlib import Path
from director.creative_director import CreativeDirector
from director.provider_factory import get_director_config_from_env, get_llm_provider_from_config
from utils.strict import require_cuda, require_real_provider

require_cuda()
director_config = get_director_config_from_env()
print("director config:", {k: v for k, v in director_config.items() if k != "provider"})

import subprocess
head = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
print("Code HEAD:", head)

# Fail loudly instead of silently echoing the OLD prompt placeholders if this
# kernel imported pre-fix code (kernels keep the old module in memory after a
# pull). Fix: Runtime -> Restart session, then Run all.
import utils.json_guard as _guard
assert hasattr(_guard, "contains_placeholder"), (
    "STALE CODE: old qwen.py loaded in this kernel. "
    "Restart the runtime (Runtime -> Restart session) and Run all."
)
print("[PASS] Kernel is running the placeholder-guard code.")

provider = get_llm_provider_from_config(director_config)
require_real_provider(provider, "Director")
print("Real director provider:", type(provider).__name__,
      "| device_resolved:", getattr(provider, "device_resolved", None))

pdir = Path(project_dir)
scene_index = json.loads((pdir / "scenes" / "scene_index.json").read_text(encoding="utf-8"))
if isinstance(scene_index, dict) and "scenes" in scene_index:
    scene_index = scene_index["scenes"]
transcript = json.loads((pdir / "transcripts" / "transcript.json").read_text(encoding="utf-8"))

t0 = time.monotonic()
director = CreativeDirector(provider=provider, memory_dir=pdir / "memory")
try:
    result = director.develop_production_plan(
        movie_metadata={"title": meta["title"], "duration_sec": 0, "source": video},
        scene_index=scene_index,
        transcript=transcript,
        num_concepts=2,
    )
except Exception as exc:
    # Surface the raw model output so a parsing failure is debuggable.
    raw = getattr(provider, "last_raw_output", None)
    print("\n[DIRECTOR] ERROR:", exc)
    if raw:
        dump = pdir / "qwen_raw_output.txt"
        dump.write_text(raw, encoding="utf-8")
        print(f"[DIRECTOR] raw model output saved -> {dump} (shown below)\n")
        print(raw[:2000])
    else:
        print("[DIRECTOR] (no raw output captured)")
    raise
dir_secs = time.monotonic() - t0

production_plan = result.get("production_plan", {})
concept = result.get("selected_concept", {})
print(f"[DIRECTOR] {dir_secs:.1f}s | thesis: {(concept.get('thesis') or '')[:90]}")
print(f"[DIRECTOR] title: {concept.get('title')} | tone: {concept.get('tone')}")
print(f"[DIRECTOR] structure sections: {len(production_plan.get('structure', [])) or 'n/a'}")

# Save the SAME director_plan.json shape the orchestrator writes, so the
# script writer/ranker/selector can consume it unchanged.
director_plan = {
    "thesis": concept.get("thesis", ""),
    "hook": concept.get("hook", ""),
    "title": concept.get("title", meta["title"]),
    "tone": concept.get("tone", ""),
    "structure": production_plan.get("structure", []),
    "scenes_to_extract": production_plan.get("scenes", []),
    "creative_generation": True,
    "concept": concept,
    "production_plan": production_plan,
    "all_concepts": result.get("generated_concepts", []),
    "director_provider": "qwen",
    "director_model": director_config.get("model"),
    "director_device": provider.device_resolved or director_config.get("device"),
}
(pdir / "director_plan.json").write_text(
    json.dumps(director_plan, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved ->", pdir / "director_plan.json")

# Free VRAM between stages; the model stays cached so cell 12 reuses it
# (no second model load -> no OOM on a 16GB T4).
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory freed (model cached for reuse by the script stage)")


In [ ]:
# ============================================================
# SCRIPT WRITER — REAL Qwen on CUDA
import json
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from scene_selection.ranker import rank_scenes
from scene_selection.selector import select_scenes
from script.qwen_writer import generate_script_qwen
from utils.strict import require_cuda

require_cuda()

# Scene ranking (thesis from the director plan) then evidence-based selection.
# This mirrors app/orchestrator.start_pipeline() exactly.
plan_data = json.loads((pdir / "director_plan.json").read_text(encoding="utf-8"))
thesis = plan_data.get("thesis") or ""
print(f"[RANK] ranking scenes for thesis: {thesis[:90]}")
rank_scenes(pdir, thesis, top_k=20)   # -> scenes/scene_ranking.json

entries = select_scenes(pdir, top_n=3)
print(f"[SELECT] {len(entries)} scene(s) -> ", pdir / "scenes" / "selected_scenes.json")

res = generate_script_qwen(
    pdir,
    model=os.environ.get("SCRIPT_MODEL", "Qwen/Qwen3-4B-Instruct-2507"),
    device="cuda",
)
sections = res.get("sections", [])
print(f"[SCRIPT] provider={res.get('script_provider')} model={res.get('script_model')} "
      f"device={res.get('script_device')}")
print(f"[SCRIPT] load={res.get('qwen_load_time_sec')}s gen={res.get('qwen_generation_time_sec')}s")
print(f"[SCRIPT] {len(sections)} section(s)")
for s in sections[:5]:
    print("   -", s.get("section_id"), "|", (s.get("text") or "")[:70])


In [ ]:
# ============================================================
# MANIFEST — prove WHAT actually ran (strict flags must be true)
manifest = {
    "notebook": "colab_qwen_validation.ipynb",
    "strict_mode": True,
    "profile": "colab-gpu",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "director_provider": "qwen",
    "director_model": os.environ.get("DIRECTOR_MODEL"),
    "director_real_generation": True,
    "script_provider": "qwen",
    "script_model": res.get("script_model"),
    "script_dtype": res.get("script_dtype"),
    "script_real_generation": True,
    "director_seconds": round(dir_secs, 2),
    "transcript_real": (json.loads((pdir / "transcripts" / "transcript.json").read_text("utf-8"))
                        .get("provider") not in (None, "none", "stub")),
}
mpath = pdir / "provider_manifest.json"
mpath.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print("\nWrote ->", mpath)

assert manifest["director_real_generation"] is True
assert manifest["script_real_generation"] is True
assert manifest["strict_mode"] is True
print("[PASS] manifest records REAL qwen generation on CUDA.")


## Validation checklist — record the result

| Check | Expected | Actually observed |
| --- | --- | --- |
| `nvidia-smi` shows a GPU | T4+ | |
| `torch.cuda.is_available()` | `True` | |
| strict prerequisites (`strict_gpu_ok`) | `True` | |
| Director provider is real Qwen | `Qwen/Qwen3-4B-Instruct-2507` @ `cuda` | |
| Director produced a thesis/structure | non-empty | |
| Script produced sections with non-empty text | yes | |
| `provider_manifest.json` flags | `director_real_generation: true`, `script_real_generation: true` | |
| Model load + generation recorded | seconds populated | |

Once all boxes are ticked, paste the manifest JSON into the ticket/commit message
and the pipeline is deemed **real-LLM validated on GPU**.

Find the artifacts at:
- `data/ColabValidation/director_plan.json`
- `data/ColabValidation/script.json`
- `data/ColabValidation/provider_manifest.json`
— and the full orchestrator flow in `app/orchestrator.start_pipeline()`.
